# 02 — FDTD physics check

Derivation **and** validation of the acoustic wave solver (`physics_solver.py`).

**No interactive visualisation here** — the real-time Pygame viewer is the visualiser
teammate's job. This notebook (1) derives the update scheme, (2) explains the wall
model, and (3) validates stability, energy behaviour, wave speed, and absorption with
a few static plots.

In [ ]:
import sys, pathlib
SRC = pathlib.Path.cwd().parent / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import matplotlib.pyplot as plt

from config import C_SOUND, FREQ, MATERIALS
from grid import RoomGrid
from room import Room
from physics_solver import WaveSolver
from utils import pressure_reflection

## 1. The update scheme

We solve the damped scalar wave equation for acoustic pressure $u(x,y,t)$:

$$\frac{\partial^2 u}{\partial t^2} + \beta\,\frac{\partial u}{\partial t} = c^2\nabla^2 u$$

Discretise **time** with 3-level central differences and **space** with the 5-point
Laplacian ($L = u_E + u_W + u_N + u_S - 4u$, and $\nabla^2u \approx L/\Delta x^2$):

$$u_{tt}\approx\frac{u^{n+1}-2u^n+u^{n-1}}{\Delta t^2},\qquad
u_t\approx\frac{u^{n+1}-u^{n-1}}{2\Delta t}.$$

Solving for the only future term $u^{n+1}$, with Courant number $r = c\,\Delta t/\Delta x$:

$$u^{n+1} = A\,u^n - B\,u^{n-1} + C\,L,\qquad
\begin{aligned}
A &= (2-\beta\Delta t)/d\\
B &= (1-\beta\Delta t/2)/d\\
C &= r^2/d
\end{aligned}\quad d = 1+\tfrac{\beta\Delta t}{2}.$$

With $\beta=0$ (our default — lossless air) this collapses to the classic leapfrog
scheme $u^{n+1}=2u^n-u^{n-1}+r^2L$. **Stability (CFL):** $r \le 1/\sqrt2 \approx 0.707$.

## 2. Walls & furniture — reflection *and* absorption

Each obstacle cell has an **absorption coefficient** $\alpha\in[0,1]$ and a target
**pressure reflection coefficient**

$$R_p = \sqrt{1-\alpha}\qquad\Rightarrow\qquad \text{absorbed energy} = 1-R_p^2 = \alpha.$$

When an air cell's neighbour is solid, the Laplacian uses a *ghost* value

$$g = u - \frac{1-R_p}{r}\,(u - u_{\text{prev}}).$$

* $R_p=1$ ($\alpha=0$): $g=u$ → rigid mirror (Neumann), **lossless**.
* $R_p=0$ ($\alpha=1$): $g=u-\tfrac1r(u-u_{\text{prev}})$ → 1st-order Mur radiating boundary, **fully absorbing**.

The $(u-u_{\text{prev}})\!\propto\! u_t$ term is essential: a purely spatial scaling
$g=R_p u$ is *reactive* (conserves energy) and cannot absorb. Coupling to the wave's
motion is what removes energy.

In [ ]:
grid = RoomGrid()
print(grid)
print(f"Courant r = {grid.r:.4f}   (CFL limit 1/sqrt(2) = {1/np.sqrt(2):.4f})")
print()
print("material -> reflection / absorption:")
for name, a in MATERIALS.items():
    Rp = float(pressure_reflection(a))
    print(f"  {name:>14}: alpha={a:.2f}  R_p={Rp:.3f}  absorbs {a*100:.0f}% of energy")

## 3. Building a room

`room.Room` stamps an `(NY, NX)` $\alpha$-map with imperative builders:
`add_border`, `add_rectangle` / `add_block`, and `add_doorway` (a carved gap).

In [ ]:
room = (Room(grid)
        .add_border("Concrete")
        .add_rectangle("Wood", 4.0, 0.0, 4.15, 6.0)   # partition wall
        .add_doorway(4.0, 2.5, 4.15, 3.5)             # 1 m doorway
        .add_block("Carpet", 1.0, 1.0, 2.5, 2.0))     # soft furniture (sofa)
print(room.summary())

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(room.alpha, origin="lower", extent=(0, grid.W, 0, grid.H), cmap="viridis")
fig.colorbar(im, ax=ax, label="absorption alpha")
ax.set_title("Room alpha-map  (0 = air)")
ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
plt.show()

## 4. Validation
### (a) Stability & energy — rigid conserves, absorbing decays

With $\beta=0$ a rigid box must **conserve** energy (the discrete leapfrog energy is
constant); absorbing walls must make it **decay**.

In [ ]:
def energy_trace(alpha_map, n=1200):
    s = WaveSolver(grid, alpha_map)
    s.add_impulse(4.0, 3.0)
    return np.array(s.run(n, record_energy=True))

E_rigid = energy_trace(Room(grid).alpha)                       # all-air -> rigid edges
E_foam  = energy_trace(Room(grid).add_border("Acoustic Foam").alpha)

t_ms = np.arange(len(E_rigid)) * grid.dt * 1e3
plt.figure(figsize=(7, 4))
plt.plot(t_ms, np.maximum(E_rigid, 1e-12), label="rigid walls (alpha=0) - conserves")
plt.plot(t_ms, np.maximum(E_foam, 1e-12),  label="Acoustic Foam walls - decays")
plt.xlabel("time (ms)"); plt.ylabel("energy  sum(u^2)"); plt.yscale("log")
plt.title("Energy vs time"); plt.legend(); plt.tight_layout(); plt.show()

half = len(E_rigid) // 2
print(f"rigid drift (2nd half): {abs(E_rigid[-1]-E_rigid[half])/E_rigid[half]*100:.2f}%")
print(f"foam retained (end/peak): {E_foam[-1]/E_foam.max():.2e}")

### (b) Wave speed — a circular wavefront travelling at $c$

In an anechoic box (an `"Open"` border, $R_p=0$) the pulse expands as a ring. The
dashed circle of radius $c\,t$ should hug the wavefront — confirming both the speed
and that propagation is isotropic (no grid-direction bias).

In [ ]:
free = WaveSolver(grid, Room(grid).add_border("Open").alpha)
free.add_impulse(4.0, 3.0, freq=FREQ)     # short pulse
free.run(55)

mx = float(np.max(np.abs(free.field))) or 1.0
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(free.field, origin="lower", extent=(0, grid.W, 0, grid.H),
               cmap="RdBu_r", vmin=-mx, vmax=mx)
ax.add_patch(plt.Circle((4.0, 3.0), C_SOUND * free.t, fill=False, ls="--", color="k"))
fig.colorbar(im, ax=ax, label="pressure")
ax.set_title(f"wavefront at t = {free.t*1e3:.1f} ms   (dashed = c*t)")
ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
plt.show()

### (c) Absorption ladder — more absorptive material, weaker echo

Retained energy (windowed mean, late/early) for a box of each material. It should
**decrease** monotonically with $\alpha$.

In [ ]:
def retained(material):
    s = WaveSolver(grid, Room(grid).add_border(material).alpha)
    s.add_impulse(4.0, 3.0)
    s.run(400)
    e0 = float(np.mean(s.run(200, record_energy=True)))
    s.run(1000)
    e1 = float(np.mean(s.run(200, record_energy=True)))
    return e1 / e0

for m in ["Concrete", "Wood", "Carpet", "Acoustic Foam"]:
    print(f"  {m:>14}: energy retained = {retained(m):.3f}")

## Summary

The solver is **stable**, **conserves energy** with rigid walls at $\beta=0$,
propagates at **$c$**, and **absorbs** at a per-material rate set by $\alpha$ — all
verified above and in `physics_solver.py`'s `__main__` suite.

The visualiser teammate builds on exactly two things from this layer:
`room.alpha` (the floor plan) and `WaveSolver.field` (the pressure to render each
frame) — plus `solver.step()` / `solver.add_impulse(x, y)` for click-to-place.